<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/01_hardware_accelerated_ml.ipynb)


# colab — Hardware Accelerated Machine Learning

## _Speeding Up ML Workloads with Free Colab GPUs_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Demonstrate dramatic speed-ups from GPU acceleration across
  classic ML, clustering, and deep-learning primitives — all on a free
  Colab T4 GPU.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Approach**: We run the same workload twice (CPU then GPU) and measure
  wall-clock time, throughput, and accuracy.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Environment**: Check the GPU and install RAPIDS `cuml-cu12`.
2. **Random Forest**: Train a 50-tree forest on 200 k samples —
  CPU scikit-learn vs GPU cuML.
3. **K-Means Clustering**: Cluster 200 k points into 20 groups — CPU vs GPU.
4. **Matrix Multiplication**: Large dense matmul (4096×4096) — PyTorch CPU vs CUDA.
5. **MLP Training**: A small neural network trained on 100 k samples — CPU vs GPU.
6. **Summary**: Side-by-side speed-up table.

### Environment Setup

We verify the GPU, install the cuML accelerator, and import the
libraries we need for all experiments.

In [ ]:
#@title Check GPU and install RAPIDS cuML
!nvidia-smi
!pip -q install cuml-cu12 --extra-index-url=https://pypi.nvidia.com
!pip -q install matplotlib

In [ ]:
#@title Imports and helpers
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

# scikit-learn (will be accelerated by cuML on GPU after we load the extension)
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Simple timing helper
def bench(label, fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} s")
    return result, elapsed

## Example 1: Random Forest Classifier

Random forests are embarrassingly parallel: every tree trains independently.
A GPU can build thousands of trees in parallel, turning a minute-long CPU
job into a sub-second one.

**Workload**: 200 k samples, 20 features, binary classification, 50 trees.

In [ ]:
#@title Generate classification dataset
N_SAMPLES = 200_000
N_FEATURES = 20
N_ESTIMATORS = 50

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=5,
    n_redundant=5,
    n_classes=2,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
#@title CPU training (scikit-learn)
clf_cpu = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    random_state=42,
    n_jobs=-1,  # use all CPU cores
)
_, t_cpu_train = bench("CPU fit", clf_cpu.fit, X_train, y_train)

y_pred_cpu = clf_cpu.predict(X_test)
acc_cpu = accuracy_score(y_test, y_pred_cpu)
print(f"CPU accuracy: {acc_cpu:.4f}")

In [ ]:
#@title Enable GPU acceleration (cuML)
%load_ext cuml.accel

In [ ]:
#@title GPU training (cuML-accelerated scikit-learn)
# The same import now transparently dispatches to the GPU.
from sklearn.ensemble import RandomForestClassifier as RF_GPU

clf_gpu = RF_GPU(
    n_estimators=N_ESTIMATORS,
    random_state=42,
)
_, t_gpu_train = bench("GPU fit", clf_gpu.fit, X_train, y_train)

y_pred_gpu = clf_gpu.predict(X_test)
acc_gpu = accuracy_score(y_test, y_pred_gpu)
print(f"GPU accuracy: {acc_gpu:.4f}")

speedup_rf = t_cpu_train / t_gpu_train
print(f"\n>>> Random Forest speed-up: {speedup_rf:.1f}×")

## Example 2: K-Means Clustering

K-Means is iterative and distance-heavy — perfect for a GPU.
Each E-step (assign points to centroids) is a massive parallel distance computation.

**Workload**: 200 k points in 10 dimensions, 20 clusters, 10 iterations.

In [ ]:
#@title Generate clustering data
N_POINTS = 200_000
N_DIMS = 10
N_CLUSTERS = 20
MAX_ITER = 10

np.random.seed(42)
centers = np.random.randn(N_CLUSTERS, N_DIMS).astype(np.float32)
labels = np.random.randint(0, N_CLUSTERS, size=N_POINTS)
X_km = centers[labels] + np.random.randn(N_POINTS, N_DIMS).astype(np.float32) * 0.3
print(f"Data shape: {X_km.shape}")

In [ ]:
#@title CPU K-Means
kmeans_cpu = KMeans(
    n_clusters=N_CLUSTERS,
    max_iter=MAX_ITER,
    n_init=1,
    random_state=42,
)
_, t_cpu_km = bench("CPU K-Means", kmeans_cpu.fit, X_km)

In [ ]:
#@title GPU K-Means (cuML-accelerated)
from sklearn.cluster import KMeans as KMeans_GPU

kmeans_gpu = KMeans_GPU(
    n_clusters=N_CLUSTERS,
    max_iter=MAX_ITER,
    n_init=1,
    random_state=42,
)
_, t_gpu_km = bench("GPU K-Means", kmeans_gpu.fit, X_km)

speedup_km = t_cpu_km / t_gpu_km
print(f"\n>>> K-Means speed-up: {speedup_km:.1f}×")

## Example 3: Matrix Multiplication

Dense matrix multiplication is the canonical GPU workload.
A T4 has thousands of CUDA cores and Tensor Cores that can sustain
hundreds of GFLOPS.

**Workload**: Multiply two 4096×4096 float32 matrices
(≈0.7 billion multiply-accumulate operations).

In [ ]:
#@title Generate large matrices
MAT_SIZE = 4096
A = torch.randn(MAT_SIZE, MAT_SIZE, dtype=torch.float32)
B = torch.randn(MAT_SIZE, MAT_SIZE, dtype=torch.float32)
print(f"Matrix size: {A.shape}  ({A.numel() * 4 / 1e9:.2f} GB each)")

In [ ]:
#@title CPU matmul
_, t_cpu_mm = bench("CPU matmul", torch.matmul, A, B)

In [ ]:
#@title GPU matmul
A_gpu = A.cuda()
B_gpu = B.cuda()
torch.cuda.synchronize()

_, t_gpu_mm = bench("GPU matmul", torch.matmul, A_gpu, B_gpu)
torch.cuda.synchronize()

speedup_mm = t_cpu_mm / t_gpu_mm
print(f"\n>>> MatMul speed-up: {speedup_mm:.1f}×")

# Rough GFLOPS estimate
flops = 2 * MAT_SIZE ** 3
print(f"Approximate throughput: {flops / t_gpu_mm / 1e12:.1f} TFLOPS (GPU)")

## Example 4: MLP Training

Neural network training is a sequence of matrix multiplications,
activation functions, and gradient updates. GPUs excel at the batched
linear algebra that dominates each step.

**Workload**: A 3-layer MLP trained for 10 epochs on 100 k synthetic
samples (20 features, 5 classes). Batch size 2048, Adam optimiser.

In [ ]:
#@title Generate synthetic tabular data
N_MLP = 100_000
F_MLP = 20
C_MLP = 5
BATCH = 2048
EPOCHS = 10

X_mlp, y_mlp = make_classification(
    n_samples=N_MLP,
    n_features=F_MLP,
    n_informative=10,
    n_redundant=5,
    n_classes=C_MLP,
    random_state=42,
)
X_mlp = torch.tensor(X_mlp, dtype=torch.float32)
y_mlp = torch.tensor(y_mlp, dtype=torch.long)

dataset = torch.utils.data.TensorDataset(X_mlp, y_mlp)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True)
print(f"Batches per epoch: {len(loader)}")

In [ ]:
#@title Define a small MLP
class MLP(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)

def train_epoch(model, loader, optim, criterion, dev):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        optim.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optim.step()
    return loss.item()

def fit(model, loader, optim, criterion, dev, epochs):
    for epoch in range(epochs):
        loss = train_epoch(model, loader, optim, criterion, dev)
    return loss

In [ ]:
#@title CPU training
model_cpu = MLP(F_MLP, 256, C_MLP)
opt_cpu = torch.optim.Adam(model_cpu.parameters(), lr=1e-3)
crit = torch.nn.CrossEntropyLoss()

_, t_cpu_mlp = bench(
    "CPU MLP train", fit, model_cpu, loader,
    opt_cpu, crit, "cpu", EPOCHS,
)

In [ ]:
#@title GPU training
model_gpu = MLP(F_MLP, 256, C_MLP).cuda()
opt_gpu = torch.optim.Adam(model_gpu.parameters(), lr=1e-3)

_, t_gpu_mlp = bench(
    "GPU MLP train", fit, model_gpu, loader,
    opt_gpu, crit, "cuda", EPOCHS,
)
torch.cuda.synchronize()

speedup_mlp = t_cpu_mlp / t_gpu_mlp
print(f"\n>>> MLP training speed-up: {speedup_mlp:.1f}×")

## Summary: Speed-Up Comparison

The table below collects the wall-clock times from every experiment.
Even a modest free T4 GPU can deliver 10–100× speed-ups on classic ML workloads.

In [ ]:
#@title Build and display results
results = pd.DataFrame({
    "Workload": [
        "Random Forest (200k, 50 trees)",
        "K-Means (200k, 20 clusters)",
        "MatMul (4096×4096)",
        "MLP train (100k, 10 epochs)",
    ],
    "CPU time (s)": [
        round(t_cpu_train, 2),
        round(t_cpu_km, 2),
        round(t_cpu_mm, 2),
        round(t_cpu_mlp, 2),
    ],
    "GPU time (s)": [
        round(t_gpu_train, 2),
        round(t_gpu_km, 2),
        round(t_gpu_mm, 2),
        round(t_gpu_mlp, 2),
    ],
})
results["Speed-up"] = results["CPU time (s)"] / results["GPU time (s)"]
results["Speed-up"] = results["Speed-up"].round(1).astype(str) + "×"

print(results.to_string(index=False))

# Simple bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
width = 0.35
ax.bar(x - width/2, results["CPU time (s)"], width, label="CPU", color="#e74c3c")
ax.bar(x + width/2, results["GPU time (s)"], width, label="GPU", color="#2ecc71")
ax.set_ylabel("Wall time (seconds, log scale)")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(results["Workload"], rotation=15, ha="right")
ax.legend()
ax.set_title("CPU vs GPU wall time")
plt.tight_layout()
plt.show()

### Exercises

1. **Scale the forest**: Increase `N_SAMPLES` to 1 M and
   `N_ESTIMATORS` to 100. Does the speed-up grow or shrink?
2. **Larger matrices**: Try `MAT_SIZE = 8192` for the matmul benchmark.
   Watch the T4 VRAM usage with `nvidia-smi`.
3. **Different clustering**: Swap K-Means for `DBSCAN` (also accelerated
   by cuML). Compare CPU vs GPU on the same 200 k point cloud.
4. **Deeper MLP**: Add a fourth hidden layer and increase `hidden` to 512.
   How does the speed-up change?
5. **Your own data**: Upload a CSV to Colab and run the Random Forest
   example on real tabular data.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
